# 🎨 AI Studio na Colabie — start jednym kliknięciem

Ten notebook uruchamia **ComfyUI** na GPU z Twojej subskrypcji **Google AI Pro**
(200 jednostek obliczeniowych miesięcznie) i podaje publiczny link do interfejsu.

**Jak używać**
1. `Środowisko wykonawcze → Zmień typ środowiska wykonawczego → GPU` (T4 / L4 (A100 tylko gdy dostępny)).
2. W komórce **2** wybierz preset modelu (albo wpisz własny).
3. `Środowisko wykonawcze → Uruchom wszystko`.
4. Po ~4–6 minutach ostatnia komórka wypisze link `https://....trycloudflare.com`.

**Pieniądze:** notebook nic nie kupuje. Zużywa wyłącznie jednostki z abonamentu AI Pro.
Gdy się skończą, Colab wraca do trybu darmowego — nie pobiera opłat z karty.


## Który GPU wybrać

| GPU | VRAM | ~jednostek/h | 200 j. wystarcza na |
|---|---|---|---|
| T4 | 16 GB | ~1,8 | ~110 h |
| L4 | 24 GB | ~4,8 | ~40 h |
| A100 40 GB | 40 GB | ~11,8 | ~17 h |

Do SDXL/Illustrious spokojnie wystarcza **T4** (najtaniej). Do FLUXa bierz **L4** (24 GB).
Jednostki ważne są 90 dni i kumulują się do ~600.


In [ ]:
#@title 1. Sprawdź maszynę { display-mode: "form" }
import subprocess, shutil, sys

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "BRAK GPU — włącz: Środowisko wykonawcze → Zmień typ środowiska → GPU")
tot, used, free = shutil.disk_usage("/content")
print(f"Dysk /content: {free/2**30:.1f} GB wolne z {tot/2**30:.1f} GB")
print(f"Python: {sys.version.split()[0]}")
try:
    import torch; print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
except Exception as e:
    print("Torch:", e)


In [ ]:
#@title 2. Konfiguracja { display-mode: "form" }
#@markdown Wybierz preset. Modele lądują na Twoim Dysku Google i zostają między sesjami.
PRESET = "flux-schnell-q4" #@param ["flux-schnell-q4", "sdxl-turbo", "illustrious-xl", "wlasny"]
USE_DRIVE = True #@param {type:"boolean"}
HF_TOKEN = "" #@param {type:"string"}
#@markdown Własny model (gdy PRESET = "wlasny"): repo HF albo bezpośredni link.
CUSTOM_REPO = "" #@param {type:"string"}
CUSTOM_FILE = "" #@param {type:"string"}
CUSTOM_SUBDIR = "checkpoints" #@param ["checkpoints", "loras", "vae", "clip", "unet", "controlnet"]
CUSTOM_URL = "" #@param {type:"string"}

PRESETS = {
    "flux-schnell-q4": [
        # (repo HF, plik, podfolder w models/)
        ("city96/FLUX.1-schnell-gguf", "flux1-schnell-Q4_K_S.gguf", "unet"),
        ("comfyanonymous/flux_text_encoders", "clip_l.safetensors", "clip"),
        ("comfyanonymous/flux_text_encoders", "t5xxl_fp8_e4m3fn.safetensors", "clip"),
        ("black-forest-labs/FLUX.1-schnell", "ae.safetensors", "vae"),
    ],
    "sdxl-turbo": [
        ("Lykon/dreamshaper-xl-v2-turbo", "DreamShaperXL_Turbo_v2_1.safetensors", "checkpoints"),
    ],
    "illustrious-xl": [
        ("OnomaAIResearch/Illustrious-XL-v2.0", "Illustrious-XL-v2.0.safetensors", "checkpoints"),
    ],
    "wlasny": [],
}
FILES = PRESETS.get(PRESET, [])
print("Preset:", PRESET, "| plików do pobrania:", len(FILES))


In [ ]:
#@title 3. Dysk Google + foldery { display-mode: "form" }
import os

ROOT = "/content/drive/MyDrive/AI-Colab" if USE_DRIVE else "/content/AI-Colab"
MODELS = f"{ROOT}/models"
for sub in ["checkpoints", "loras", "vae", "clip", "unet", "controlnet"]:
    os.makedirs(f"{MODELS}/{sub}", exist_ok=True)
os.makedirs(f"{ROOT}/output", exist_ok=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
print("Modele:", MODELS)


In [ ]:
#@title 4. ComfyUI + cloudflared (2–4 min) { display-mode: "form" }
import os, subprocess, sys

if not os.path.isdir("/content/ComfyUI/.git"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/comfyanonymous/ComfyUI", "/content/ComfyUI"], check=True)
os.chdir("/content/ComfyUI")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

CF = "/usr/local/bin/cloudflared"
if not os.path.exists(CF):
    subprocess.run(["wget", "-q", "-O", CF,
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    os.chmod(CF, 0o755)

# modele z Dysku widoczne dla ComfyUI (bez kopiowania)
extra = f"""ai_colab:
    base_path: {MODELS}
    checkpoints: checkpoints
    loras: loras
    vae: vae
    clip: clip
    unet: unet
    controlnet: controlnet
"""
open("/content/ComfyUI/extra_model_paths.yaml", "w").write(extra)
print("OK")


In [ ]:
#@title 5. Pobierz model (raz — zostaje na Dysku) { display-mode: "form" }
import os, time

def have(path, min_mb=50):
    return os.path.exists(path) and os.path.getsize(path) > min_mb * 1024**2

def fetch_hf(repo, filename, subdir):
    target = f"{MODELS}/{subdir}/{os.path.basename(filename)}"
    if have(target):
        print("✔ już jest:", target); return target
    from huggingface_hub import hf_hub_download
    t0 = time.time()
    path = hf_hub_download(repo_id=repo, filename=filename, local_dir=f"{MODELS}/{subdir}",
                           token=HF_TOKEN or None, resume_download=True)
    print(f"✔ {filename}: {os.path.getsize(path)/2**30:.2f} GB w {time.time()-t0:.0f}s")
    return path

def fetch_url(url, subdir, filename):
    import requests
    target = f"{MODELS}/{subdir}/{filename}"
    if have(target):
        print("✔ już jest:", target); return target
    head = {"range": f"bytes={os.path.getsize(target)}-"} if os.path.exists(target) else {}
    with requests.get(url, headers=head, stream=True, timeout=60) as r:
        r.raise_for_status()
        mode = "ab" if head else "wb"
        with open(target, mode) as f:
            for chunk in r.iter_content(8 * 1024**2):
                f.write(chunk)
    print("✔", target, f"{os.path.getsize(target)/2**30:.2f} GB")
    return target

for repo, filename, subdir in FILES:
    fetch_hf(repo, filename, subdir)

if CUSTOM_REPO and CUSTOM_FILE:
    fetch_hf(CUSTOM_REPO, CUSTOM_FILE, CUSTOM_SUBDIR)
elif CUSTOM_URL:
    fetch_url(CUSTOM_URL, CUSTOM_SUBDIR, CUSTOM_FILE or CUSTOM_URL.split("/")[-1].split("?")[0])


In [ ]:
#@title 6. Uruchom ComfyUI + publiczny link { display-mode: "form" }
import os, re, subprocess, sys, time

LOG = "/content/comfyui.log"
open(LOG, "w").close()
proc = subprocess.Popen(
    [sys.executable, "main.py", "--listen", "0.0.0.0", "--port", "8188",
     "--preview-method", "auto", "--output-directory", f"{ROOT}/output"],
    cwd="/content/ComfyUI", stdout=open(LOG, "a"), stderr=subprocess.STDOUT)

for _ in range(180):
    if "To see the GUI go to" in open(LOG, errors="ignore").read():
        break
    if proc.poll() is not None:
        print(open(LOG, errors="ignore").read()[-3000:]); raise SystemExit("ComfyUI padł — log wyżej")
    time.sleep(2)
print("ComfyUI działa lokalnie na :8188")

TLOG = "/content/tunnel.log"
open(TLOG, "w").close()
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188",
                  "--no-autoupdate", "--logfile", TLOG, "--loglevel", "info"])
url = None
for _ in range(90):
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open(TLOG, errors="ignore").read())
    if m:
        url = m.group(0); break
    time.sleep(1)
print("\n=== OTWÓRZ:", url or "brak linku — sprawdź /content/tunnel.log", "===")


## Jak dodać własny model

- **Hugging Face:** ustaw `PRESET = "wlasny"`, wpisz `CUSTOM_REPO` (np. `autor/repo`) i `CUSTOM_FILE` (np. `model.safetensors`).
- **Civitai / direct link:** zostaw `CUSTOM_REPO` puste, wpisz `CUSTOM_URL` i `CUSTOM_FILE`.
- **LoRA:** `CUSTOM_SUBDIR = "loras"`.
- Repo „gated" (np. Illustrious v0.1) wymaga `HF_TOKEN` z https://huggingface.co/settings/tokens.

Pliki zostają w `MyDrive/AI-Colab/models/...`, więc kolejna sesja pobiera tylko ComfyUI (2–4 min), nie modele.
